<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB06 &mdash; Valuation & Recipe Configuration</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Build the valuation recipe (with the FX rule), enable instrument event configuration, and run valuations grouped by instrument name and sub-holding key at the 2024-09-30 anchor date.</div>
</div>

<sub>IBOR pack sequence: NB00 &nbsp;&rarr;&nbsp; NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; **NB06** &nbsp;&rarr;&nbsp; NB07 &nbsp;&rarr;&nbsp; NB08</sub>

# NB06: Valuation & Recipe Configuration

Scope: `ibor-training-v8`

NB00 already creates the OTC instruments (IRS, Term Deposit) and the zero coupon bond as SimpleInstruments, and loads all FX and MTM quotes. This notebook therefore only: builds the valuation recipe (with the FX rule), enables instrument event configuration, determines the latest priceable date, and runs valuations grouped by name + SubHoldingKey.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import os, json
from datetime import datetime, timedelta, timezone
import lusid as lu
import lusid.models as lm

secrets_path = "secrets.json"
with open(secrets_path) as f:
    secrets = json.load(f)

lusid_factory = lu.extensions.SyncApiClientFactory(
    config_loaders=[lu.extensions.ArgsConfigurationLoader(
        api_url=secrets["api"]["lusidUrl"], access_token=secrets["api"]["accessToken"])])

recipe_api = lusid_factory.build(lu.ConfigurationRecipeApi)
tp_api = lusid_factory.build(lu.TransactionPortfoliosApi)
valuation_api = lusid_factory.build(lu.AggregationApi)
instruments_api = lusid_factory.build(lu.InstrumentsApi)
quotes_api = lusid_factory.build(lu.QuotesApi)
portfolios_api = lusid_factory.build(lu.PortfoliosApi)

SCOPE = 'ibor-training-v8'
QUOTE_SCOPE = 'ibor-training-v8-quotes'
print("Connected.")

---
## Create Valuation Recipe

Five market data rules: ClientInternal Price, RIC Price (Client), RIC Rate (Lusid), LusidInstrumentId Price, and the FX rule (`Fx.*.*`). The FX rule plus `suppliers={"Fx": "Client"}` is essential for the iShares portfolios (IBOR-AITECH, IBOR-BLKC, IBOR-GAGG) which hold instruments denominated in TWD, HKD, CNY, JPY etc. and need FX resolution to USD. `allow_any_instruments_with_sec_uid_to_price_off_lookup=True` forces quote-based pricing.

In [ ]:
recipe = lm.ConfigurationRecipe(
    scope=SCOPE,
    code="IBOR-VALUATION-RECIPE",
    market=lm.MarketContext(
        market_rules=[
            lm.MarketDataKeyRule(
                key="Quote.ClientInternal.*", supplier="Client",
                data_scope=QUOTE_SCOPE, quote_type="Price", field="mid", quote_interval="5D"),
            lm.MarketDataKeyRule(
                key="Quote.RIC.*", supplier="Client",
                data_scope=QUOTE_SCOPE, quote_type="Price", field="mid", quote_interval="5D"),
            lm.MarketDataKeyRule(
                key="Quote.RIC.*", supplier="Lusid",
                data_scope=QUOTE_SCOPE, quote_type="Rate", field="mid", quote_interval="5D"),
            lm.MarketDataKeyRule(
                key="Quote.LusidInstrumentId.*", supplier="Client",
                data_scope=QUOTE_SCOPE, quote_type="Price", field="mid", quote_interval="5D"),
            lm.MarketDataKeyRule(
                key="Fx.*.*", supplier="Client",
                data_scope=QUOTE_SCOPE, quote_type="Price", field="mid", quote_interval="5D"),
        ],
        suppliers={"Fx": "Client"},
        options=lm.MarketOptions(
            default_supplier="Client",
            default_instrument_code_type="ClientInternal",
            default_scope=QUOTE_SCOPE)),
    pricing=lm.PricingContext(
        options=lm.PricingOptions(
            allow_any_instruments_with_sec_uid_to_price_off_lookup=True)))

recipe_api.upsert_configuration_recipe(
    upsert_recipe_request=lm.UpsertRecipeRequest(configuration_recipe=recipe))
print(f"Created recipe: {SCOPE}/IBOR-VALUATION-RECIPE")
print("  5 rules: ClientInternal Price, RIC Price (Client), RIC Rate (Lusid), LUID Price, Fx.*.* (Client)")
print("  suppliers={Fx: Client}, allow_any_instruments_with_sec_uid_to_price_off_lookup=True")

---
## Enable InstrumentEventConfiguration

Lets LUSID generate lifecycle events (coupons, maturities) on the relevant portfolios.

In [ ]:
for pf_code in ["IBOR-FI", "IBOR-EQ", "IBOR-MA", "IBOR-GAGG"]:
    try:
        pf = portfolios_api.get_portfolio(scope=SCOPE, code=pf_code)
        portfolios_api.update_portfolio(scope=SCOPE, code=pf_code,
            update_portfolio_request=lm.UpdatePortfolioRequest(
                display_name=pf.display_name,
                description=pf.description or "",
                instrument_event_configuration=lm.InstrumentEventConfiguration(
                    transaction_template_scopes=[SCOPE],
                    recipe_id=lm.ResourceId(scope=SCOPE, code="IBOR-VALUATION-RECIPE"))))
        print(f"  Enabled InstrumentEventConfiguration on {pf_code}")
    except Exception as e:
        print(f"  {pf_code}: {str(e)[:100]}")

---
## Run Valuations

Values all seven portfolios at `EFFECTIVE_AT`, the latest date for which quotes exist (set in the cell above). The grouping includes `Holding/default/SubHoldingKey` as well as instrument name: the portfolios use SubHoldingKeys (Strategy + CustodianAccount), and an instrument that spans two SHK buckets collapses to a null PV if you group by instrument name alone. Grouping by name + SHK keeps every lot resolvable. Expected totals: IBOR-EQ ~$12.0M, IBOR-FI ~$69.3M, IBOR-MA ~$31.9M, IBOR-SP500 ~$50.2M, IBOR-AITECH ~$10.3M, IBOR-BLKC ~$5.1M, IBOR-GAGG ~$130.9M.

---
## Determine Valuation Date (latest available)

Rather than hardcoding a date or using `datetime.now()` (which fails because no quotes exist that recently), the valuation date is set to the most recent date for which equity Price quotes exist in the quote scope. This gives a "latest available" valuation that always resolves and never goes stale, without widening the recipe's quote interval. The probe walks month-ends backward from today and stops at the first date where a representative equity (NVDA) has a quote within the recipe's 5D interval.

In [ ]:
# Valuation date.
# The data window runs across 2024-2025 with full holdings and quote coverage at
# 2024-09-30. A "latest priceable date" probe is unreliable on this dataset because
# near-empty early dates trivially succeed (an empty portfolio raises no pricing error),
# so we anchor to the date where every portfolio is fully populated and priced.
EFFECTIVE_AT = "2024-09-30T00:00:00+00:00"
print(f"Valuation date: {EFFECTIVE_AT[:10]}")

In [ ]:
print(f"Valuing all portfolios at {EFFECTIVE_AT[:10]}")
portfolios = ["IBOR-EQ", "IBOR-FI", "IBOR-MA", "IBOR-SP500", "IBOR-AITECH", "IBOR-BLKC", "IBOR-GAGG"]

for pf in portfolios:
    try:
        result = valuation_api.get_valuation(
            valuation_request=lm.ValuationRequest(
                recipe_id=lm.ResourceId(scope=SCOPE, code="IBOR-VALUATION-RECIPE"),
                metrics=[lm.AggregateSpec(key="Holding/default/PV", op="Value")],
                group_by=["Instrument/default/Name", "Holding/default/SubHoldingKey"],
                portfolio_entity_ids=[
                    lm.PortfolioEntityId(scope=SCOPE, code=pf, portfolio_entity_type="SinglePortfolio")],
                valuation_schedule=lm.ValuationSchedule(effective_at=EFFECTIVE_AT)))

        positions = 0
        total_mv = 0
        errs = []
        for row in result.data:
            pv = row.get("Holding/default/PV")
            err = row.get("Holding/default/PV/ResultValueType")
            if err == "ResultError":
                errs.append(row.get("Holding/default/PV/Error", "unknown"))
            elif pv is not None:
                positions += 1
                total_mv += pv

        print(f"  {pf}: {positions} lines valued, total MV = {total_mv:,.0f}")
        if errs:
            print(f"    {len(errs)} errors. First: {str(errs[0])[:150]}")
    except Exception as e:
        print(f"  {pf}: ERROR {str(e)[:200]}")

print()
print("NB06 Complete. Proceed to NB07: Reconciliation.")